# Polars: The Modern DataFrame Library

Polars is rapidly replacing pandas in production data pipelines. The reasons are compelling:

- **10 to 100x faster** than pandas on most operations due to Rust internals and SIMD vectorization
- **Lazy evaluation**: build a query plan, let the optimizer run the minimum work needed
- **Native parallelism**: automatically uses all CPU cores with no extra configuration
- **Rust-backed**: memory safe, no GIL, and predictable performance
- **Apache Arrow memory format**: zero-copy interop with DuckDB, PyArrow, and most modern data tools

By the end of this notebook you will know the Polars expression API, lazy evaluation, groupby aggregations, string and date operations, joins, window functions, and how to integrate Polars into a scikit-learn ML pipeline.

In [ ]:
import polars as pl
import numpy as np

print(f"Polars version: {pl.__version__}")
print(f"NumPy version:  {np.__version__}")

## 1. Core Data Structures

Polars has two primary data structures:

| Structure | Description |
|-----------|-------------|
| `pl.Series` | A single typed column (like a typed array) |
| `pl.DataFrame` | A collection of named Series columns |

Unlike pandas, every column in a Polars DataFrame has a strict, enforced dtype. There is no mixed-type `object` dtype used for strings; strings are `pl.Utf8` (also called `pl.String`). This strictness is what enables Polars to compile optimized execution plans.

In [ ]:
# Create a sample DataFrame
df = pl.DataFrame({
    "name":       ["Alice", "Bob", "Carol", "David", "Eve", "Frank", "Grace"],
    "age":        [29, 34, 27, 45, 31, 38, 26],
    "salary":     [72000, 95000, 61000, 110000, 83000, 98000, 57000],
    "department": ["Engineering", "Sales", "Engineering", "Management", "Sales", "Engineering", "HR"],
    "score":      [88.5, 76.2, 91.0, 84.3, 79.8, 93.1, 70.5],
})

print("DataFrame:")
print(df)
print("\nShape:", df.shape)
print("\nDtypes:")
print(df.dtypes)
print("\nSchema:")
print(df.schema)
print("\nDescribe:")
print(df.describe())

## 2. Reading and Writing Data

Polars supports fast I/O for CSV, Parquet, JSON, Arrow IPC, and more. The key distinction:

- **`pl.read_csv()`** reads eagerly (loads the full file into a DataFrame immediately)
- **`pl.scan_csv()`** reads lazily (returns a `LazyFrame` that only reads what is needed when `.collect()` is called)

For large files, always prefer `scan_*` functions. Polars will push down filters and select only the columns it needs.

In [ ]:
import tempfile, os

# Write a temp CSV file
csv_path = "/tmp/polars_demo.csv"
df.write_csv(csv_path)
print("CSV written to", csv_path)

# Eager read
df_from_csv = pl.read_csv(csv_path)
print("\nEager CSV read:")
print(df_from_csv.head(3))

# Write and read Parquet
parquet_path = "/tmp/polars_demo.parquet"
df.write_parquet(parquet_path)
df_from_parquet = pl.read_parquet(parquet_path)
print("\nParquet read:")
print(df_from_parquet.head(3))

# Lazy scan (returns a LazyFrame, not data)
lazy_df = pl.scan_csv(csv_path)
print("\nLazy scan type:", type(lazy_df))
print("Collect to get data:")
print(lazy_df.filter(pl.col("salary") > 80000).collect())

## 3. Expressions: The Core of Polars

The Polars **expression API** is the most important concept to learn. Instead of chaining methods on a DataFrame (pandas style), you build **expressions** using `pl.col()` and pass them to DataFrame methods.

Core methods:
- `.select([exprs])` - choose and transform columns
- `.filter(expr)` - filter rows by a boolean expression
- `.with_columns([exprs])` - add or update columns
- `.sort(by)` - sort rows
- `.rename({old: new})` - rename columns

Expressions are **lazy by nature**: they describe what to do, not when to do it. This lets Polars batch multiple column operations into a single parallel pass.

In [ ]:
# select: choose columns and apply transformations
print("Selected columns with transformation:")
print(df.select([
    pl.col("name"),
    pl.col("salary"),
    (pl.col("salary") / 1000).alias("salary_k"),
]))

# filter: keep rows matching a condition
print("\nFilter salary > 80000 AND score > 80:")
print(df.filter((pl.col("salary") > 80000) & (pl.col("score") > 80)))

# with_columns: add new columns (does not mutate; returns a new DataFrame)
print("\nAdd bonus column (10% of salary):")
df_with_bonus = df.with_columns(
    (pl.col("salary") * 0.10).alias("bonus")
)
print(df_with_bonus)

# sort
print("\nSort by salary descending:")
print(df.sort("salary", descending=True))

# rename
print("\nRename 'score' to 'perf_score':")
print(df.rename({"score": "perf_score"}).head(3))

## 4. Lazy Evaluation and Query Optimization

**Eager execution** (pandas default): every operation runs immediately and allocates intermediate memory.

**Lazy execution** (Polars LazyFrame): operations are recorded into a logical query plan. When you call `.collect()`, Polars:
1. Optimizes the plan (predicate pushdown, projection pushdown, common subexpression elimination)
2. Compiles to a physical plan
3. Executes in parallel across CPU cores

Use `.explain()` to see the optimized query plan before executing.

In [ ]:
# Build a lazy query chain
lazy_query = (
    pl.scan_csv("/tmp/polars_demo.csv")
    .filter(pl.col("age") > 28)
    .with_columns(
        (pl.col("salary") * 1.05).alias("salary_adjusted")
    )
    .select(["name", "department", "salary", "salary_adjusted", "score"])
    .sort("salary_adjusted", descending=True)
)

print("Optimized query plan:")
print(lazy_query.explain())

print("\nCollected result:")
result = lazy_query.collect()
print(result)

## 5. GroupBy and Aggregations

Polars `group_by().agg()` runs all aggregations in a single parallel pass. In pandas, chaining multiple `.agg()` functions forces multiple group scans. In Polars, all aggregations are computed simultaneously.

Common aggregation expressions:
- `pl.col("x").sum()`, `.mean()`, `.std()`, `.min()`, `.max()`
- `pl.col("x").count()`, `.n_unique()`
- `pl.col("x").first()`, `.last()`

In [ ]:
# GroupBy with multiple simultaneous aggregations
agg_result = (
    df
    .group_by("department")
    .agg([
        pl.col("salary").sum().alias("total_salary"),
        pl.col("salary").mean().alias("avg_salary"),
        pl.col("salary").min().alias("min_salary"),
        pl.col("salary").max().alias("max_salary"),
        pl.col("salary").std().alias("std_salary"),
        pl.col("score").mean().alias("avg_score"),
        pl.col("name").count().alias("headcount"),
    ])
    .sort("avg_salary", descending=True)
)
print("GroupBy department:")
print(agg_result)

# GroupBy multiple columns
df2 = df.with_columns(pl.Series("region", ["West", "East", "West", "East", "West", "East", "West"]))
print("\nGroupBy department + region:")
print(
    df2.group_by(["department", "region"])
    .agg(pl.col("salary").mean().alias("avg_salary"))
    .sort(["department", "region"])
)

## 6. String and Date Operations

Polars provides vectorized string and date operations via the `.str` and `.dt` namespaces.

**String namespace** (`pl.col("x").str.*`):
- `.contains(pattern)`, `.starts_with()`, `.ends_with()`
- `.replace(pattern, replacement)`, `.replace_all()`
- `.split(by)`, `.strip_chars()`, `.to_uppercase()`, `.to_lowercase()`

**Date namespace** (`pl.col("x").dt.*`):
- `.year()`, `.month()`, `.day()`, `.weekday()`
- `.strftime(format)`, date arithmetic with `pl.duration()`

In [ ]:
from datetime import date

# String operations
str_df = pl.DataFrame({
    "full_name": ["Alice Smith", "Bob Jones", "Carol White", "David Brown"],
    "email":     ["alice@example.com", "bob@corp.org", "carol@example.com", "david@startup.io"],
})

print("String operations:")
print(
    str_df.with_columns([
        pl.col("full_name").str.to_uppercase().alias("upper_name"),
        pl.col("full_name").str.split(" ").list.first().alias("first_name"),
        pl.col("email").str.contains("example").alias("is_example"),
        pl.col("email").str.replace(r"@.*", "").alias("username"),
    ])
)

# Date operations
date_df = pl.DataFrame({
    "hire_date": [
        date(2021, 3, 15), date(2019, 11, 1),
        date(2023, 7, 22), date(2020, 5, 30),
    ],
    "name": ["Alice", "Bob", "Carol", "David"],
})

print("\nDate operations:")
print(
    date_df.with_columns([
        pl.col("hire_date").dt.year().alias("hire_year"),
        pl.col("hire_date").dt.month().alias("hire_month"),
        pl.col("hire_date").dt.weekday().alias("weekday"),
        (pl.col("hire_date") + pl.duration(days=365)).alias("first_review"),
    ])
)

## 7. Joins in Polars

Polars supports all standard join types. The syntax is uniform across all join types and all joins run in parallel.

| Join type | Description |
|-----------|-------------|
| `inner`   | Only matching rows from both sides |
| `left`    | All rows from left, matching from right |
| `full`    | All rows from both sides |
| `semi`    | Left rows that have a match on the right (filter only) |
| `anti`    | Left rows that do NOT have a match on the right |

In [ ]:
employees = pl.DataFrame({
    "emp_id":     [1, 2, 3, 4, 5],
    "name":       ["Alice", "Bob", "Carol", "David", "Eve"],
    "dept_id":    [10, 20, 10, 30, 20],
})

departments = pl.DataFrame({
    "dept_id":   [10, 20, 40],
    "dept_name": ["Engineering", "Sales", "Finance"],
})

print("Inner join (only matching dept_id):")
print(employees.join(departments, on="dept_id", how="inner"))

print("\nLeft join (all employees, null where no dept):")
print(employees.join(departments, on="dept_id", how="left"))

print("\nFull outer join:")
print(employees.join(departments, on="dept_id", how="full"))

print("\nSemi join (employees who have a matching dept):")
print(employees.join(departments, on="dept_id", how="semi"))

print("\nAnti join (employees with NO matching dept):")
print(employees.join(departments, on="dept_id", how="anti"))

## 8. Window Functions

Window functions compute aggregations over a group of rows without collapsing the DataFrame. In Polars the `.over()` expression defines the partition window.

This is analogous to SQL `OVER (PARTITION BY ...)`. Common uses:
- Rank employees within their department
- Compute cumulative sums per group
- Compute rolling averages within a time partition

In [ ]:
sales_df = pl.DataFrame({
    "employee":   ["Alice", "Bob", "Carol", "David", "Eve", "Frank", "Grace", "Heidi"],
    "department": ["Eng", "Sales", "Eng", "Sales", "Eng", "Sales", "HR", "HR"],
    "revenue":    [12000, 9500, 15000, 11000, 8000, 13500, 7000, 9000],
    "quarter":    [1, 1, 1, 1, 2, 2, 2, 2],
})

result = sales_df.with_columns([
    # Rank within department by revenue (highest = rank 1)
    pl.col("revenue").rank(descending=True).over("department").alias("dept_rank"),
    # Department total revenue (broadcast back to each row)
    pl.col("revenue").sum().over("department").alias("dept_total"),
    # Percentage of department total
    (pl.col("revenue") / pl.col("revenue").sum().over("department") * 100)
        .round(1)
        .alias("pct_of_dept"),
    # Cumulative revenue within department (sorted by revenue)
    pl.col("revenue").cum_sum().over("department").alias("cumsum_dept"),
])

print("Window functions:")
print(result.sort(["department", "dept_rank"]))

## 9. Polars vs Pandas: Performance Comparison

The most convincing demonstration of Polars is a side-by-side benchmark. We create a 1 million row dataset and run a multi-column groupby aggregation in both pandas and Polars. Expect Polars to be 5 to 50x faster depending on hardware and the specific operation.

In [ ]:
import pandas as pd
import time

N = 1_000_000
rng = np.random.default_rng(42)

departments = ["Engineering", "Sales", "Marketing", "HR", "Finance"]
regions     = ["North", "South", "East", "West"]

data = {
    "department": rng.choice(departments, N),
    "region":     rng.choice(regions, N),
    "salary":     rng.integers(40000, 150000, N).astype(float),
    "score":      rng.uniform(50, 100, N),
    "tenure":     rng.integers(1, 20, N),
}

# Pandas benchmark
pdf = pd.DataFrame(data)
t0 = time.perf_counter()
_ = pdf.groupby(["department", "region"]).agg(
    avg_salary=("salary", "mean"),
    total_salary=("salary", "sum"),
    avg_score=("score", "mean"),
    count=("salary", "count"),
).reset_index()
pandas_time = time.perf_counter() - t0

# Polars benchmark
pldf = pl.DataFrame(data)
t0 = time.perf_counter()
_ = (
    pldf
    .group_by(["department", "region"])
    .agg([
        pl.col("salary").mean().alias("avg_salary"),
        pl.col("salary").sum().alias("total_salary"),
        pl.col("score").mean().alias("avg_score"),
        pl.col("salary").count().alias("count"),
    ])
    .sort(["department", "region"])
)
polars_time = time.perf_counter() - t0

speedup = pandas_time / polars_time
print(f"Dataset: {N:,} rows")
print(f"Pandas:  {pandas_time:.4f}s")
print(f"Polars:  {polars_time:.4f}s")
print(f"Speedup: {speedup:.1f}x faster")

## 10. Polars with Scikit-learn and ML Pipelines

Polars integrates with the broader Python ML ecosystem via:
- `.to_numpy()` converts a DataFrame or Series to a NumPy array
- `.to_pandas()` converts to a pandas DataFrame when a library requires it

Because Polars uses Apache Arrow memory format internally, these conversions are usually zero-copy (no data is duplicated), making them fast even for large datasets.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Build a clean Polars DataFrame
ml_df = pl.DataFrame({
    "age":        rng.integers(22, 60, 500).astype(float),
    "salary":     rng.integers(40000, 150000, 500).astype(float),
    "score":      rng.uniform(50, 100, 500),
    "tenure":     rng.integers(1, 20, 500).astype(float),
    "promoted":   rng.integers(0, 2, 500),   # binary target
})

# Feature engineering with Polars expressions
ml_df = ml_df.with_columns([
    (pl.col("salary") / 1000).alias("salary_k"),
    (pl.col("score") / 100).alias("score_norm"),
])

feature_cols = ["age", "salary_k", "score_norm", "tenure"]

# Convert Polars to NumPy for scikit-learn
X = ml_df.select(feature_cols).to_numpy()   # shape: (500, 4)
y = ml_df["promoted"].to_numpy()             # shape: (500,)

print(f"X shape: {X.shape}, dtype: {X.dtype}")
print(f"y shape: {y.shape}, dtype: {y.dtype}")

# Standard ML workflow
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_s, y_train)
preds = model.predict(X_test_s)

print(f"\nLogistic Regression accuracy: {accuracy_score(y_test, preds):.3f}")
print("Polars to NumPy conversion is zero-copy when memory layout allows.")

## 11. When to Use Polars vs Pandas

Polars is not a drop-in replacement for every pandas use case. Some libraries (matplotlib, seaborn, scikit-learn) still expect pandas DataFrames or NumPy arrays. Use `.to_pandas()` as the bridge.

**Use Polars when:**
- Dataset is larger than ~100MB and pandas is slow
- You need reproducible, optimized data pipelines
- Working with Parquet files in production
- You want lazy evaluation to avoid redundant computation

**Stick with pandas when:**
- Ecosystem requires it (some AutoML, time-series libraries)
- Existing codebase is pandas-heavy and migration cost is high
- Dataset is small (under 10MB) and iteration speed is the priority

In [ ]:
# Comparison table as a Polars DataFrame
comparison = pl.DataFrame({
    "Feature":         ["API Style", "Speed", "Memory", "Lazy Eval", "Ecosystem", "Learning Curve", "Production Use"],
    "Pandas":          [
        "Method chaining, inplace mutation",
        "Baseline (single-threaded)",
        "High (copies are common)",
        "No (all eager)",
        "Excellent (sklearn, matplotlib, etc)",
        "Low (familiar to most)",
        "Common but slow at scale",
    ],
    "Polars":          [
        "Expression API, immutable DataFrames",
        "10 to 100x faster (multi-threaded)",
        "Low (Arrow format, fewer copies)",
        "Yes (LazyFrame)",
        "Growing (use .to_pandas() as bridge)",
        "Medium (new expression API to learn)",
        "Rapidly adopted in data engineering",
    ],
})

with pl.Config(tbl_rows=20, tbl_width_chars=120):
    print(comparison)

## Key Takeaways

1. **Expression API**: Use `pl.col()` expressions with `.select()`, `.filter()`, `.with_columns()`. Never mutate a DataFrame in place.

2. **Lazy first**: Prefer `pl.scan_*` over `pl.read_*` for large files. Chain operations, then call `.collect()` once.

3. **Parallelism is automatic**: Polars uses all CPU cores by default. You get parallel execution without any configuration.

4. **Strict types**: Every column has a strict dtype. No `object` dtype. This enables compile-time optimization and prevents silent type coercion bugs.

5. **Arrow interop**: `.to_numpy()` and `.to_pandas()` are nearly zero-copy, making Polars a drop-in data source for scikit-learn, PyTorch, and other ML libraries.

6. **GroupBy is parallel**: All aggregations in a single `.agg()` call run concurrently. Chain as many as you need without performance penalty.

**Next step**: Notebook 10 covers DuckDB, which lets you run SQL directly on Polars DataFrames, pandas DataFrames, and Parquet files without any server setup.